# ICU Monitor Analysis — MIMIC-IV & eICU
## Research: ICU Monitor Energy Optimization
**Author:** Konstantinos Dinos Kritsis
**Date:** April 2026

This notebook covers the analysis pipeline:
1. Loading reference files (d_items and hospital)
2. Filtering chartevents to vital signs only
3. Building the hourly monitoring activity timeline
4. Data cleaning

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


## Step 1 — Loading d_items.csv.gz (MIMIC Dictionary)
This file translates itemid number codes into human readable vital sign names.
Without this file we cannot know what chartevents measurements actually are.

In [2]:
# Loading d_items — the MIMIC dictionary file
df_items = pd.read_csv('/Users/dinoskritsis/Desktop/MIMIC/d_items.csv.gz',
                        compression='gzip')

print("=== d_items — MIMIC Dictionary ===")
print("Total items:", len(df_items))
print("\nColumns:", df_items.columns.tolist())
print("\nFirst 5 rows:")
df_items.head()

=== d_items — MIMIC Dictionary ===
Total items: 4095

Columns: ['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname', 'param_type', 'lownormalvalue', 'highnormalvalue']

First 5 rows:


,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


In [3]:
# Filtering d_items to only vital signs relevant to monitor activity
vital_sign_categories = ['Routine Vital Signs']

vital_signs = df_items[df_items['category'].isin(vital_sign_categories)]

print("=== Vital Sign Items — Routine Vital Signs ===")
print("Total vital sign items found:", len(vital_signs))
print("\nAll vital sign items:")
print(vital_signs[['itemid', 'label', 'category', 'unitname']].to_string())

=== Vital Sign Items — Routine Vital Signs ===
Total vital sign items found: 50

All vital sign items:
      itemid                                  label             category unitname
2     220045                             Heart Rate  Routine Vital Signs      bpm
5     220048                           Heart Rhythm  Routine Vital Signs      NaN
6     220050       Arterial Blood Pressure systolic  Routine Vital Signs     mmHg
7     220051      Arterial Blood Pressure diastolic  Routine Vital Signs     mmHg
8     220052           Arterial Blood Pressure mean  Routine Vital Signs     mmHg
24    220179   Non Invasive Blood Pressure systolic  Routine Vital Signs     mmHg
25    220180  Non Invasive Blood Pressure diastolic  Routine Vital Signs     mmHg
26    220181       Non Invasive Blood Pressure mean  Routine Vital Signs     mmHg
337   223761                 Temperature Fahrenheit  Routine Vital Signs       °F
338   223762                    Temperature Celsius  Routine Vital Signs     

In [4]:
# Defining the core monitor activity itemids
# These are the vital signs a Philips bedside monitor tracks continuously
core_monitor_items = [
    220045,  # Heart Rate
    220050,  # Arterial Blood Pressure systolic
    220051,  # Arterial Blood Pressure diastolic
    220052,  # Arterial Blood Pressure mean
    220179,  # Non Invasive Blood Pressure systolic
    220180,  # Non Invasive Blood Pressure diastolic
    220181,  # Non Invasive Blood Pressure mean
    220210,  # Respiratory Rate
    220277,  # SpO2 (Oxygen Saturation)
    223761,  # Temperature Fahrenheit
    223762,  # Temperature Celsius
]

# Filtering d_items to only these core items
core_items = df_items[df_items['itemid'].isin(core_monitor_items)]

print("=== Core Monitor Activity Items ===")
print(core_items[['itemid', 'label', 'unitname']].to_string())
print(f"\nTotal core items: {len(core_items)}")

=== Core Monitor Activity Items ===
     itemid                                  label  unitname
2    220045                             Heart Rate       bpm
6    220050       Arterial Blood Pressure systolic      mmHg
7    220051      Arterial Blood Pressure diastolic      mmHg
8    220052           Arterial Blood Pressure mean      mmHg
24   220179   Non Invasive Blood Pressure systolic      mmHg
25   220180  Non Invasive Blood Pressure diastolic      mmHg
26   220181       Non Invasive Blood Pressure mean      mmHg
28   220210                       Respiratory Rate  insp/min
36   220277            O2 saturation pulseoxymetry         %
337  223761                 Temperature Fahrenheit        °F
338  223762                    Temperature Celsius        °C

Total core items: 11


## Step 2 — Filtering Chartevents to Core Monitor Items
Now we filter the full chartevents file to keep only the 11 core vital sign items.
This reduces the file from 330+ million rows to only the rows relevant to monitor activity.
Note: We load in chunks because the full file is too large to load all at once.

In [5]:
# Filtering chartevents to core monitor items only
# Using chunked reading because the full file is ~30GB uncompressed

print("Filtering chartevents to core monitor items...")
print("This will take several minutes — please wait...\n")

chunk_size = 1_000_000  # Read 1 million rows at a time
chunks = []

for i, chunk in enumerate(pd.read_csv(
    '/Users/dinoskritsis/Desktop/MIMIC/chartevents.csv.gz',
    compression='gzip',
    chunksize=chunk_size
)):
    # Keep only rows where itemid is one of our core monitor items
    filtered = chunk[chunk['itemid'].isin(core_monitor_items)]
    chunks.append(filtered)
    
    if i % 10 == 0:
        print(f"Processed {(i+1) * chunk_size:,} rows so far...")

# Combining all filtered chunks
df_monitor_events = pd.concat(chunks, ignore_index=True)

print(f"\nDone!")
print(f"Total monitor events found: {len(df_monitor_events):,}")
print(f"Unique ICU stays with monitor events: {df_monitor_events['stay_id'].nunique():,}")
print(f"\nFirst 5 rows:")
df_monitor_events.head()

Filtering chartevents to core monitor items...
This will take several minutes — please wait...

Processed 1,000,000 rows so far...
Processed 11,000,000 rows so far...
Processed 21,000,000 rows so far...
Processed 31,000,000 rows so far...
Processed 41,000,000 rows so far...
Processed 51,000,000 rows so far...
Processed 61,000,000 rows so far...
Processed 71,000,000 rows so far...
Processed 81,000,000 rows so far...
Processed 91,000,000 rows so far...
Processed 101,000,000 rows so far...
Processed 111,000,000 rows so far...
Processed 121,000,000 rows so far...
Processed 131,000,000 rows so far...
Processed 141,000,000 rows so far...
Processed 151,000,000 rows so far...
Processed 161,000,000 rows so far...
Processed 171,000,000 rows so far...
Processed 181,000,000 rows so far...
Processed 191,000,000 rows so far...
Processed 201,000,000 rows so far...
Processed 211,000,000 rows so far...
Processed 221,000,000 rows so far...
Processed 231,000,000 rows so far...
Processed 241,000,000 rows 

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,10000032,29079034,39553978,18704.0,2180-07-23 14:00:00,2180-07-23 14:20:00,223761,98.7,98.7,°F,0.0
1,10000032,29079034,39553978,18704.0,2180-07-23 14:11:00,2180-07-23 14:17:00,220179,84,84.0,mmHg,0.0
2,10000032,29079034,39553978,18704.0,2180-07-23 14:11:00,2180-07-23 14:17:00,220180,48,48.0,mmHg,0.0
3,10000032,29079034,39553978,18704.0,2180-07-23 14:11:00,2180-07-23 14:17:00,220181,56,56.0,mmHg,0.0
4,10000032,29079034,39553978,18704.0,2180-07-23 14:12:00,2180-07-23 14:17:00,220045,91,91.0,bpm,0.0


In [6]:
# Saving the filtered monitor events to a CSV file
# So we never have to re-process the full chartevents again

save_path = '/Users/dinoskritsis/Desktop/MIMIC/monitor_events_filtered.csv.gz'

df_monitor_events.to_csv(save_path, compression='gzip', index=False)

print(f"Saved successfully!")
print(f"File saved to: {save_path}")
print(f"Total rows saved: {len(df_monitor_events):,}")

Saved successfully!
File saved to: /Users/dinoskritsis/Desktop/MIMIC/monitor_events_filtered.csv.gz
Total rows saved: 53,806,854


## Step 3 — Loading hospital.csv.gz (eICU Hospital Context)
This file provides metadata about each of the 208 hospitals in the eICU dataset.
It will be used for cross-hospital energy comparisons.

In [7]:
# Loading eICU hospital file
df_hospital = pd.read_csv('/Users/dinoskritsis/Desktop/E-ICU/hospital.csv.gz',
                           compression='gzip')

print("=== eICU Hospital File ===")
print("Total hospitals:", len(df_hospital))
print("\nColumns:", df_hospital.columns.tolist())
print("\nFirst 5 rows:")
df_hospital.head()

=== eICU Hospital File ===
Total hospitals: 208

Columns: ['hospitalid', 'numbedscategory', 'teachingstatus', 'region']

First 5 rows:


,hospitalid,numbedscategory,teachingstatus,region
0,56,<100,f,Midwest
1,58,100 - 249,f,Midwest
2,59,<100,f,Midwest
3,60,<100,f,Midwest
4,61,<100,f,Midwest


In [8]:
# Hospital size distribution
print("=== Hospital Size Distribution ===")
print(df_hospital['numbedscategory'].value_counts())

print("\n=== Hospital Region Distribution ===")
print(df_hospital['region'].value_counts())

=== Hospital Size Distribution ===
numbedscategory
100 - 249    62
<100         46
250 - 499    35
>= 500       23
Name: count, dtype: int64

=== Hospital Region Distribution ===
region
Midwest      70
South        56
West         43
Northeast    13
Name: count, dtype: int64


## Step 4 — Trying to Build the Hourly Monitoring Activity Timeline (MIMIC-IV)
For every ICU stay we will create an hourly signal showing:
- ACTIVE: a vital sign was recorded in that hour
- IDLE: no recording during an occupied bed
- EMPTY: bed is empty (between patients)

This timeline is the foundation of the entire energy analysis.

In [10]:
# Loading  icustays into this notebook. i WILL NEED IT FOR NEXT STEP
df_icustays = pd.read_csv('/Users/dinoskritsis/Desktop/MIMIC/icustays.csv.gz',
                           compression='gzip')

df_icustays['intime'] = pd.to_datetime(df_icustays['intime'])
df_icustays['outtime'] = pd.to_datetime(df_icustays['outtime'])

print("icustays loaded successfully!")
print("Total stays:", len(df_icustays))

icustays loaded successfully!
Total stays: 94458


In [13]:
# First  I need to clean icustays — remove records with missing outtime
df_icustays_clean = df_icustays.dropna(subset=['intime', 'outtime']).copy()

print(f"Original stays: {len(df_icustays):,}")
print(f"After removing missing times: {len(df_icustays_clean):,}")
print(f"Removed: {len(df_icustays) - len(df_icustays_clean)} records")

# Also remove stays shorter than 4 hours
df_icustays_clean['los_hours'] = (
    df_icustays_clean['outtime'] - df_icustays_clean['intime']
).dt.total_seconds() / 3600

df_icustays_clean = df_icustays_clean[df_icustays_clean['los_hours'] >= 4]

print(f"After removing stays under 4 hours: {len(df_icustays_clean):,}")
print(f"\nCleaned dataset ready!")

Original stays: 94,458
After removing missing times: 94,444
Removed: 14 records
After removing stays under 4 hours: 93,730

Cleaned dataset ready!


In [11]:
# Loading the filtered monitor events we saved earlier
df_monitor_events = pd.read_csv(
    '/Users/dinoskritsis/Desktop/MIMIC/monitor_events_filtered.csv.gz',
    compression='gzip',
    parse_dates=['charttime']
)

# Also make sure icustays timestamps are parsed as dates
df_icustays['intime'] = pd.to_datetime(df_icustays['intime'])
df_icustays['outtime'] = pd.to_datetime(df_icustays['outtime'])

print("=== Files Loaded ===")
print(f"Monitor events: {len(df_monitor_events):,} rows")
print(f"ICU stays: {len(df_icustays):,} rows")
print(f"\nMonitor events date range:")
print(f"Earliest: {df_monitor_events['charttime'].min()}")
print(f"Latest:   {df_monitor_events['charttime'].max()}")

=== Files Loaded ===
Monitor events: 53,806,854 rows
ICU stays: 94,458 rows

Monitor events date range:
Earliest: 2110-01-11 12:42:00
Latest:   2214-07-26 16:00:00


In [15]:
# Building the hourly monitoring activity timeline for each ICU stay
# Using the CLEANED dataset with no missing times

print("Building hourly activity timeline...")
print(f"Processing {len(df_icustays_clean):,} stays...\n")

results = []

for idx, stay in df_icustays_clean.iterrows():
    stay_id = stay['stay_id']
    intime = stay['intime']
    outtime = stay['outtime']
    los_hours = stay['los_hours']

    # Skip if either time is NaT just in case
    if pd.isnull(intime) or pd.isnull(outtime):
        continue

    # Get monitor events for this stay
    stay_events = df_monitor_events[
        df_monitor_events['stay_id'] == stay_id
    ]

    # Create hourly bins
    try:
        hours = pd.date_range(
            start=intime.floor('h'),
            end=outtime.ceil('h'),
            freq='h'
        )
    except Exception:
        continue

    for hour in hours:
        hour_end = hour + pd.Timedelta(hours=1)
        events_in_hour = stay_events[
            (stay_events['charttime'] >= hour) &
            (stay_events['charttime'] < hour_end)
        ]
        status = 'ACTIVE' if len(events_in_hour) > 0 else 'IDLE'

        results.append({
            'stay_id': stay_id,
            'hour': hour,
            'status': status,
            'los_hours': los_hours,
            'careunit': stay['first_careunit']
        })

df_timeline = pd.DataFrame(results)

print(f"Timeline built successfully!")
print(f"Total hours analysed: {len(df_timeline):,}")
print(f"\nActivity breakdown:")
print(df_timeline['status'].value_counts())

Building hourly activity timeline...
Processing 93,730 stays...



KeyboardInterrupt: 

In [16]:
# FAST version because the previous one took for ever to load I waited for 2 hours and nothing happened to I will use pandas groupby instead of a loop
print("Building hourly activity timeline (fast version)...")

# Round charttime down to the nearest hour
df_monitor_events['hour'] = df_monitor_events['charttime'].dt.floor('h')

# Get one row per stay per hour where monitor was ACTIVE
active_hours = df_monitor_events.groupby(
    ['stay_id', 'hour']
).size().reset_index(name='event_count')

active_hours['status'] = 'ACTIVE'

print(f"Total ACTIVE monitor hours found: {len(active_hours):,}")
print(f"Unique stays with activity: {active_hours['stay_id'].nunique():,}")
print(f"\nFirst 5 rows:")
active_hours.head()

Building hourly activity timeline (fast version)...
Total ACTIVE monitor hours found: 7,926,044
Unique stays with activity: 94,438

First 5 rows:


,stay_id,hour,event_count,status
0,30000153,2174-09-29 12:00:00,7,ACTIVE
1,30000153,2174-09-29 13:00:00,8,ACTIVE
2,30000153,2174-09-29 14:00:00,7,ACTIVE
3,30000153,2174-09-29 15:00:00,6,ACTIVE
4,30000153,2174-09-29 16:00:00,7,ACTIVE


In [17]:
# Now  I will calculate the IDLE hours, when the patient was in the ICU but the monitor had no recordings.
# These are hours where the patient was present but no monitor recording

# Merge active hours with icustays to get full occupancy picture
active_hours_with_stay = active_hours.merge(
    df_icustays_clean[['stay_id', 'intime', 'outtime', 
                        'los_hours', 'first_careunit']],
    on='stay_id', how='left'
)

# Calculate total occupied hours per stay
total_occupied = df_icustays_clean.copy()
total_occupied['total_hours'] = total_occupied['los_hours'].round().astype(int)

# Active hours per stay
active_per_stay = active_hours.groupby('stay_id').size().reset_index(name='active_hours')

# Merge to get idle hours
stay_summary = total_occupied.merge(active_per_stay, on='stay_id', how='left')
stay_summary['active_hours'] = stay_summary['active_hours'].fillna(0)
stay_summary['idle_hours'] = stay_summary['total_hours'] - stay_summary['active_hours']
stay_summary['idle_hours'] = stay_summary['idle_hours'].clip(lower=0)

print("=== Monitor Activity Summary per Stay ===")
print(f"Total ICU stays analysed: {len(stay_summary):,}")
print(f"\nTotal occupied hours: {stay_summary['total_hours'].sum():,.0f}")
print(f"Total ACTIVE hours:   {stay_summary['active_hours'].sum():,.0f}")
print(f"Total IDLE hours:     {stay_summary['idle_hours'].sum():,.0f}")

active_pct = stay_summary['active_hours'].sum() / stay_summary['total_hours'].sum() * 100
idle_pct = stay_summary['idle_hours'].sum() / stay_summary['total_hours'].sum() * 100

print(f"\nACTIVE: {active_pct:.1f}% of all occupied hours")
print(f"IDLE:   {idle_pct:.1f}% of all occupied hours")

=== Monitor Activity Summary per Stay ===
Total ICU stays analysed: 93,730

Total occupied hours: 8,226,347
Total ACTIVE hours:   7,921,402
Total IDLE hours:     322,431

ACTIVE: 96.3% of all occupied hours
IDLE:   3.9% of all occupied hours


In [18]:
# Energy calculations
POWER_MAIN = 0.0352  # kW (35.2W — Philips IntelliVue main value)
POWER_LOW = 0.012    # kW (12W — low power mode)
POWER_HIGH = 0.0443  # kW (44.3W — high end sensitivity)
CO2_FACTOR = 0.28    # kg CO2 per kWh

total_hours = stay_summary['total_hours'].sum()
idle_hours = stay_summary['idle_hours'].sum()
active_hours_total = stay_summary['active_hours'].sum()

# Baseline energy — monitor running at full power all the time
baseline_energy_kwh = total_hours * POWER_MAIN

# Wasted energy — idle hours at full power
wasted_energy_kwh = idle_hours * POWER_MAIN

# Potential saving — if idle hours used low power mode instead
saved_energy_kwh = idle_hours * (POWER_MAIN - POWER_LOW)

# CO2 calculations
baseline_co2 = baseline_energy_kwh * CO2_FACTOR
wasted_co2 = wasted_energy_kwh * CO2_FACTOR
saved_co2 = saved_energy_kwh * CO2_FACTOR

print("=" * 50)
print("ENERGY ANALYSIS — MIMIC-IV")
print("=" * 50)
print(f"\nBaseline energy (full power always): {baseline_energy_kwh:,.0f} kWh")
print(f"Wasted energy (idle hours):          {wasted_energy_kwh:,.0f} kWh")
print(f"Potential energy saving:             {saved_energy_kwh:,.0f} kWh")
print(f"Saving percentage:                   {saved_energy_kwh/baseline_energy_kwh*100:.1f}%")
print(f"\nBaseline CO2:  {baseline_co2:,.0f} kg")
print(f"Wasted CO2:    {wasted_co2:,.0f} kg")
print(f"Saveable CO2:  {saved_co2:,.0f} kg")

print(f"\n=== Sensitivity Analysis ===")
for power, label in [(POWER_LOW, '12W low'), (POWER_MAIN, '35.2W main'), (POWER_HIGH, '44.3W high')]:
    energy = total_hours * power
    print(f"{label}: {energy:,.0f} kWh | CO2: {energy*CO2_FACTOR:,.0f} kg")

ENERGY ANALYSIS — MIMIC-IV

Baseline energy (full power always): 289,567 kWh
Wasted energy (idle hours):          11,350 kWh
Potential energy saving:             7,480 kWh
Saving percentage:                   2.6%

Baseline CO2:  81,079 kg
Wasted CO2:    3,178 kg
Saveable CO2:  2,095 kg

=== Sensitivity Analysis ===
12W low: 98,716 kWh | CO2: 27,641 kg
35.2W main: 289,567 kWh | CO2: 81,079 kg
44.3W high: 364,427 kWh | CO2: 102,040 kg


In [19]:
# Saving the stay summary for further analysis
stay_summary.to_csv(
    '/Users/dinoskritsis/Desktop/MIMIC/stay_summary.csv.gz',
    compression='gzip', index=False
)

print("Stay summary saved!")
print(f"\n=== KEY RESULTS SUMMARY ===")
print(f"Dataset: MIMIC-IV — {len(stay_summary):,} ICU stays")
print(f"Total monitor hours: {total_hours:,}")
print(f"Idle hours: {idle_hours:,} ({idle_hours/total_hours*100:.1f}%)")
print(f"Wasted energy: {wasted_energy_kwh:,.0f} kWh")
print(f"Saveable energy: {saved_energy_kwh:,.0f} kWh ({saved_energy_kwh/baseline_energy_kwh*100:.1f}%)")
print(f"Saveable CO2: {saved_co2:,.0f} kg")
print(f"\nThese results are based on the CONSERVATIVE scenario only.")
print(f"The moderate and aggressive AI scenarios will produce higher savings.")

Stay summary saved!

=== KEY RESULTS SUMMARY ===
Dataset: MIMIC-IV — 93,730 ICU stays
Total monitor hours: 8,226,347
Idle hours: 322,431.0 (3.9%)
Wasted energy: 11,350 kWh
Saveable energy: 7,480 kWh (2.6%)
Saveable CO2: 2,095 kg

These results are based on the CONSERVATIVE scenario only.
The moderate and aggressive AI scenarios will produce higher savings.


In [20]:
# Energy breakdown by ICU unit type
print("=== Energy Analysis by ICU Unit Type ===\n")

unit_summary = stay_summary.groupby('first_careunit').agg(
    total_stays=('stay_id', 'count'),
    total_hours=('total_hours', 'sum'),
    active_hours=('active_hours', 'sum'),
    idle_hours=('idle_hours', 'sum')
).reset_index()

# Calculate energy and CO2 per unit
unit_summary['wasted_energy_kwh'] = unit_summary['idle_hours'] * POWER_MAIN
unit_summary['saveable_energy_kwh'] = unit_summary['idle_hours'] * (POWER_MAIN - POWER_LOW)
unit_summary['saveable_co2_kg'] = unit_summary['saveable_energy_kwh'] * CO2_FACTOR
unit_summary['idle_pct'] = (unit_summary['idle_hours'] / unit_summary['total_hours'] * 100).round(1)

# Sort by wasted energy
unit_summary = unit_summary.sort_values('wasted_energy_kwh', ascending=False)

print(unit_summary[[
    'first_careunit', 'total_stays', 'total_hours',
    'idle_hours', 'idle_pct', 'wasted_energy_kwh', 'saveable_co2_kg'
]].to_string(index=False))

=== Energy Analysis by ICU Unit Type ===

                                  first_careunit  total_stays  total_hours  idle_hours  idle_pct  wasted_energy_kwh  saveable_co2_kg
                              Neuro Intermediate         5758       696030     98655.0      14.2          3472.6560       640.862880
    Cardiac Vascular Intensive Care Unit (CVICU)        14684      1175625     45486.0       3.9          1601.1072       295.477056
              Medical Intensive Care Unit (MICU)        20546      1866588     44718.0       2.4          1574.0736       290.488128
Medical/Surgical Intensive Care Unit (MICU/SICU)        15318      1145624     32558.0       2.8          1146.0416       211.496768
             Surgical Intensive Care Unit (SICU)        12921      1217909     29494.0       2.4          1038.1888       191.593024
                        Coronary Care Unit (CCU)        10660       799624     23195.0       2.9           816.4640       150.674720
                           

In [21]:
# Time of day analysis — when are idle periods most common?
print("=== Time of Day Analysis ===\n")

# Add hour of day to active hours
active_hours['hour_of_day'] = active_hours['hour'].dt.hour

# Count active hours by hour of day
active_by_hour = active_hours.groupby('hour_of_day').size().reset_index(
    name='active_hours_count'
)

# Total possible hours per hour of day
# (total stays × 1 hour per slot)
total_by_hour = stay_summary['total_hours'].sum() / 24

active_by_hour['idle_estimate'] = total_by_hour - active_by_hour['active_hours_count']
active_by_hour['idle_pct'] = (
    active_by_hour['idle_estimate'] / total_by_hour * 100
).round(1)

print("Hour of day | Active hours | Est. Idle % ")
print("-" * 45)
for _, row in active_by_hour.iterrows():
    bar = "█" * int(row['idle_pct'])
    print(f"  {int(row['hour_of_day']):02d}:00      | "
          f"{int(row['active_hours_count']):,}      | "
          f"{row['idle_pct']}% {bar}")

=== Time of Day Analysis ===

Hour of day | Active hours | Est. Idle % 
---------------------------------------------
  00:00      | 329,280      | 3.9% ███
  01:00      | 322,931      | 5.8% █████
  02:00      | 331,656      | 3.2% ███
  03:00      | 327,882      | 4.3% ████
  04:00      | 338,673      | 1.2% █
  05:00      | 331,130      | 3.4% ███
  06:00      | 333,498      | 2.7% ██
  07:00      | 328,718      | 4.1% ████
  08:00      | 343,175      | -0.1% 
  09:00      | 333,740      | 2.6% ██
  10:00      | 340,588      | 0.6% 
  11:00      | 334,855      | 2.3% ██
  12:00      | 344,495      | -0.5% 
  13:00      | 334,079      | 2.5% ██
  14:00      | 339,339      | 1.0% █
  15:00      | 330,961      | 3.4% ███
  16:00      | 336,767      | 1.7% █
  17:00      | 323,270      | 5.7% █████
  18:00      | 319,063      | 6.9% ██████
  19:00      | 313,759      | 8.5% ████████
  20:00      | 327,971      | 4.3% ████
  21:00      | 318,094      | 7.2% ███████
  22:00      | 323,653

Note: I went a little of track trying to see what would come up. Althought the eICU cleaning and timeline were completed after the initial MIMIC energy analysis. All results presented above use the cleaned MIMIC-IV dataset. eICU energy analysis follows below.

## Note on Notebook Structure

In research, I believe the analysis is often iterative. The MIMIC-IV energy calculations above 
were run first to validate the approach. The eICU cleaning, eICU timeline, and 
formal cleaning decisions log follow below in their correct logical order.

All MIMIC-IV results above use the fully cleaned dataset (stays under 4 hours 
and missing discharge times removed). The eICU analysis follows the same 
cleaning rules applied below.

## Step 5 — Data Cleaning: eICU
Removing records with negative length of stay and stays under 4 hours.
These cleaning rules are consistent with the MIMIC-IV cleaning applied earlier.

In [ ]:
# Loading eICU patient file for cleaning
df_patient = pd.read_csv('/Users/dinoskritsis/Desktop/E-ICU/patient.csv.gz',
                          compression='gzip')

print(f"Original eICU stays: {len(df_patient):,}")

# Calculate length of stay in hours
df_patient['los_hours'] = df_patient['unitdischargeoffset'] / 60

# Remove negative length of stay
df_patient_clean = df_patient[df_patient['los_hours'] >= 0].copy()
print(f"After removing negative LOS: {len(df_patient_clean):,}")

# Remove stays under 4 hours
df_patient_clean = df_patient_clean[df_patient_clean['los_hours'] >= 4].copy()
print(f"After removing stays under 4 hours: {len(df_patient_clean):,}")

removed = len(df_patient) - len(df_patient_clean)
print(f"\nTotal records removed: {removed:,}")
print(f"Final cleaned eICU dataset: {len(df_patient_clean):,} stays")

Original eICU stays: 200,859
After removing negative LOS: 200,857
After removing stays under 4 hours: 188,789

Total records removed: 12,070
Final cleaned eICU dataset: 188,789 stays


## Step 4b — Building Hourly Activity Timeline: eICU
Using vitalPeriodic which was recorded directly by Philips monitors at 5-minute intervals.
observationoffset is in minutes since admission — converted to hours.
Only heartrate and sao2 used as activity signals (99.5% and 90.6% present respectively).

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries reloaded successfully!")

Libraries reloaded successfully!


In [4]:
# Building  eICU activity timeline using vitalPeriodic
print("Loading vitalPeriodic — please wait, large file...")

df_periodic_full = pd.read_csv(
    '/Users/dinoskritsis/Desktop/E-ICU/vitalPeriodic.csv.gz',
    compression='gzip',
    usecols=['patientunitstayid', 'observationoffset', 'heartrate', 'sao2']
)

print(f"Loaded: {len(df_periodic_full):,} rows")

# Keep only rows with at least one valid reading
df_periodic_full = df_periodic_full.dropna(subset=['heartrate', 'sao2'], how='all')
print(f"After removing rows with no heartrate or sao2: {len(df_periodic_full):,}")

# Convert offset minutes to hours
df_periodic_full['hour_offset'] = (df_periodic_full['observationoffset'] // 60).astype(int)

# Remove negative offsets
df_periodic_full = df_periodic_full[df_periodic_full['hour_offset'] >= 0]
print(f"After removing negative offsets: {len(df_periodic_full):,}")

# Get active hours per stay
active_hours_eicu = df_periodic_full.groupby(
    ['patientunitstayid', 'hour_offset']
).size().reset_index(name='event_count')

active_hours_eicu['status'] = 'ACTIVE'

print(f"\nTotal ACTIVE monitor hours in eICU: {len(active_hours_eicu):,}")
print(f"Unique stays with activity: {active_hours_eicu['patientunitstayid'].nunique():,}")
print(f"\nFirst 5 rows:")
active_hours_eicu.head()

Loading vitalPeriodic — please wait, large file...
Loaded: 146,671,642 rows
After removing rows with no heartrate or sao2: 146,575,663
After removing negative offsets: 146,446,254

Total ACTIVE monitor hours in eICU: 12,466,089
Unique stays with activity: 192,706

First 5 rows:


,patientunitstayid,hour_offset,event_count,status
0,141168,1,1,ACTIVE
1,141168,2,12,ACTIVE
2,141168,3,12,ACTIVE
3,141168,4,12,ACTIVE
4,141168,5,12,ACTIVE


## Step 6 — Data Cleaning Decisions Log

All cleaning decisions documented here for the thesis methodology chapter.

### MIMIC-IV Cleaning Decisions
1. **Removed 14 records** with missing outtime — discharge time not recorded
2. **Removed stays under 4 hours** — too short to represent meaningful monitoring
3. **Filtered chartevents to 11 core vital sign itemids** — only items directly 
   representing Philips monitor activity retained
4. **Final MIMIC-IV dataset: 93,730 stays**

### eICU Cleaning Decisions
1. **Removed 2 records** with negative length of stay — discharge before admission
2. **Removed 12,068 stays under 4 hours** — consistent with MIMIC-IV rule
3. **Removed vitalPeriodic rows with no heartrate or sao2** — 95,979 rows removed
4. **Removed negative observation offsets** — 129,409 rows removed
5. **Final eICU dataset: 188,789 stays**

### Justification for 4-Hour Threshold
Stays under 4 hours are excluded because they represent brief observation 
periods rather than full ICU admissions. This threshold is consistent with 
existing ICU data processing literature.

### Key Assumption
Monitor activity is inferred from vital sign recordings — not from direct 
device telemetry. A gap in recordings during an occupied bed is classified 
as IDLE. This assumption and its implications are acknowledged in the 
thesis limitations section.